In [ ]:
#| default_exp core

# API

> API for ipykernel-helper

In [ ]:
#| export
from fastcore.meta import delegates
from fastcore.utils import patch,dict2obj
from types import ModuleType, FunctionType, MethodType, BuiltinFunctionType
from inspect import signature, currentframe
from functools import cmp_to_key,partial
from collections.abc import Mapping
from textwrap import dedent
from cloudscraper import create_scraper
from toolslm.funccall import *
from ast import literal_eval
from urllib.parse import urlparse, urljoin

import typing,warnings,re,os,html2text

from IPython.core.interactiveshell import InteractiveShell
from IPython.core.completer import ProvisionalCompleterWarning
from jedi import Interpreter, Script as jscript

from IPython.core.display import DisplayObject
from IPython.display import display,Markdown,HTML
from IPython.core.oinspect import Inspector

In [ ]:
from fastcore.test import *

In [ ]:
#| export
warnings.filterwarnings('ignore', category=ProvisionalCompleterWarning)

In [ ]:
from pprint import pprint

In [ ]:
#| export
def _safe_repr(obj, max_len=200):
    "Safely get the repr() of an object, truncating if it exceeds max_len."
    try:
        s = str(obj)
        return s[:max_len] + ("…" if len(s)>max_len else "")
    except Exception as e: return f"<repr error: {str(e)}>"

In [ ]:
s = "Some long string that will be truncated"
print(_safe_repr(s, max_len=20))

Some long string tha…


In [ ]:
o = dict(name="Example", data=[1,2,3,4,5] * 5, nested={"a": 1, "b": 2, "c": [3, 4, 5] * 10})
print(_safe_repr(o, max_len=40))

{'name': 'Example', 'data': [1, 2, 3, 4,…


In [ ]:
#| export
@patch
def user_items(self:InteractiveShell, max_len=200, xtra_skip=()):
    "Get user-defined vars & funcs from namespace."
    ns,nsh = self.user_ns,self.user_ns_hidden
    ignore = {'nbmeta', 'receive_nbmeta'}
    ignore.add(xtra_skip)
    rm_types = (
        type, FunctionType, ModuleType, MethodType, BuiltinFunctionType,
        getattr(typing, '_SpecialGenericAlias', ()),
        getattr(typing, '_GenericAlias', ()),
        getattr(typing, '_SpecialForm', ())
    )
    user_items = {k:v for k, v in ns.items()
                  if not k in ignore and k not in nsh}
    user_vars = {k:_safe_repr(v, max_len=max_len)
                 for k, v in user_items.items() if not k.startswith('_') and not isinstance(v, rm_types)}
    user_fns = {k:str(signature(v)) for k, v in user_items.items()
                if isinstance(v, FunctionType) and v.__module__ == '__main__' and not k.startswith('__')}
    return user_vars,user_fns

In [ ]:
ipy = get_ipython()
_vs,_fs = ipy.user_items()
pprint(_vs)
print('---')
pprint(_fs)

{'TEST_IMAGE': 'images/puppy.jpg',
 'TEST_IMAGE_BW': 'images/mnist3.png',
 'custom_types': "{<class 'pathlib.Path'>}",
 'exception': '<fastcore.test.ExceptionExpected object>',
 'ipy': '<ipykernel.zmqshell.ZMQInteractiveShell object>',
 'o': "{'name': 'Example', 'data': [1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, "
      "5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5], 'nested': {'a': 1, 'b': 2, 'c': [3, "
      '4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5,…',
 's': 'Some long string that will be truncated',
 'user_items': 'None'}
---
{'_safe_repr': '(obj, max_len=200)'}


In [ ]:
#| export
def _rank(c, s):
    "Rank a completion `c` for text `s` with namespace `ns`."
    parts = s.split('.')
    is_public = not c.text.startswith('_')
    if c.type=='param': r=1
    elif c.mod=='__main__': r=2 # local
    elif len(parts)>1 and parts[0]==c.mod: r=3 # module
    elif c.mod=='builtins': r=4
    else: r=5
    return r if is_public else r+0.1

In [ ]:
#| export
@patch
def ranked_complete(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    lines = code.splitlines(True)
    if line_no: offset = sum(len(lines[i]) for i in range(line_no-1)) + col_no -1
    else: offset = len(code)
    cs = self.Completer.completions(code, offset)
    def _c(a):
        res = dict2obj({attr: getattr(a, attr) for attr in dir(a) if attr[0]!='_'})
        res['mod']= getattr(ns.get(a.text, None), '__module__', None)
        res['rank'] = _rank(res, s=code)
        return res
    # Remove dunder vars, unless the user seems to be looking for them explicitly
    return [_c(c) for c in cs if not c.text.startswith('__') or '__' in code]

In [ ]:
from random import random

In [ ]:
def range_ex(
    a:str # some param
):
    "some func docstring"
    ...
ipy.ranked_complete('rang')

[{'end': 4,
  'signature': '',
  'start': 0,
  'text': 'range',
  'type': 'class',
  'mod': None,
  'rank': 5},
 {'end': 4,
  'signature': '(a: str)',
  'start': 0,
  'text': 'range_ex',
  'type': 'function',
  'mod': '__main__',
  'rank': 2}]

In [ ]:
res = ipy.ranked_complete('a="foo"\na.', 2, 3)
res[:2]

[{'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'capitalize',
  'type': 'function',
  'mod': None,
  'rank': 5},
 {'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'casefold',
  'type': 'function',
  'mod': None,
  'rank': 5}]

In [ ]:
#| export
def _signatures(ns, s, line, col):
    ctx = Interpreter(s, [ns]).get_signatures(line, col)
    if not ctx: ctx = jscript(s).get_signatures(line, col)
    return ctx

@patch
def sig_help(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    ctx = _signatures(ns, code, line=line_no, col=col_no)
    def _s(s): return {'label':s.description,'typ':s.type, 'mod':s.module_name, 'doc':s.docstring(),
                       'idx':s.index, 'params':[{'name':p.name, 'desc':p.description} for p in s.params]}
    return [_s(opt) for opt in ctx]

In [ ]:
s = 'range('
res = ipy.sig_help(s, 1, len(s))
res[0]

{'label': 'class range',
 'typ': 'class',
 'mod': 'builtins',
 'doc': 'range(stop: int)\nrange(start: int, stop: int, step: int=...)\n\nrange(stop) -> range object\nrange(start, stop[, step]) -> range object\n\nReturn an object that produces a sequence of integers from start (inclusive)\nto stop (exclusive) by step.  range(i, j) produces i, i+1, i+2, ..., j-1.\nstart defaults to 0, and stop is omitted!  range(4) produces 0, 1, 2, 3.\nThese are exactly the valid indices for a list of 4 elements.\nWhen step is given, it specifies the increment (or decrement).',
 'idx': 0,
 'params': [{'name': 'stop', 'desc': 'param stop: int'}]}

In [ ]:
#| export
@patch
def get_vars(self:InteractiveShell, vs:list, literal=True):
    "Get variables from namespace."
    ns = self.user_ns
    def _maybe_eval(o):
        try: literal_eval(repr(o)); return o
        except: return str(o)
    return {v:_maybe_eval(ns[v]) if literal else str(ns[v]) for v in vs if v in ns}

In [ ]:
x, y, fp = 3, 4, open('./00_core.ipynb')

In [ ]:
test_eq(ipy.get_vars(['x', 'y', 'fp']).values(), [x,y,str(fp)])
test_eq(ipy.get_vars(['x', 'y', 'fp'], False).values(), [str(x),str(y),str(fp)])

In [ ]:
#| export
def _get_schema(ns: dict, t):
    "Check if tool `t` has errors."
    if t not in ns: return f"`{t}` not found. Did you run it?"
    try: return {'type':'function', 'function':get_schema(ns[t], pname='parameters')}
    except Exception as e: return f"`{t}`: {e}."

@patch
def get_schemas(self:InteractiveShell, fs:list):
    "Get schemas from namespace."
    ns = self.user_ns
    return {f:_get_schema(ns,f) for f in fs}

In [ ]:
ipy.get_schemas(['range_ex'])

{'range_ex': {'name': 'range_ex',
  'description': 'some func docstring',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'string', 'description': 'some param'}},
   'required': ['a']}}}

Errors are passed back as strings:

In [ ]:
def add(a:int,b:int): return a + b
ipy.get_schemas(['add'])

{'add': '`add`: Docstring missing!.'}

In [ ]:
ipy.get_schemas(['div'])

{'div': '`div` not found. Did you run it?'}

In [ ]:
#| export
@patch
def xpush(self:InteractiveShell, interactive=False, **kw):
    "Like `push`, but with kwargs"
    self.push(kw, interactive=interactive)

In [ ]:
ipy.push(dict(a=2))
a

2

In [ ]:
# ipykernel_helper version uses `**kwargs`
ipy.xpush(a=3)
a

3

The main benefits of using `ipy.push(dict(a=2))` over directly executing code are:

1. **Bulk variable assignment** - You can set multiple variables at once with a single command
2. **Programmatic variable injection** - It provides a way to inject variables into the namespace from another context or function
3. **No execution history** - Variables are added without creating an entry in the execution history
4. **No side effects** - It's a "pure" namespace modification without executing any code that might have side effects

There are several interesting functions in the IPython interpreter object that are useful for notebook development and interactive computing:

1. **`reset`/`reset_selective`** - Clear variables from the namespace (either all or selectively)
2. **`run_cell`/`run_cell_async`** - Execute code in a cell programmatically
3. **`set_next_input`** - Programmatically set the content of the next cell
4. **`system`/`system_raw`/`system_piped`** - Execute shell commands with different output handling
5. **`run_line_magic`/`run_cell_magic`** - Execute IPython magics programmatically
6. **`set_custom_exc`** - Set custom exception handlers

### Displaying MIME data

In [ ]:
cts = '#### A heading\n\nThis is **bold**.'
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

#### A heading

This is **bold**.

In [ ]:
#| export
@patch
def publish(self:InteractiveShell, data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    if isinstance(data, DisplayObject): data,_ = self.display_formatter.format(data)
    elif not isinstance(data, Mapping): data = {f'{mimetype}/{subtype}': data}
    self.display_pub.publish(data, metadata=meta, transient=kw, update=update)

In [ ]:
ipy.publish(cts, 'markdown', foo='bar')

#### A heading

This is **bold**.

In [ ]:
ipy.publish(HTML('<b>hi</b> there'))

In [ ]:
ipy.publish({'text/plain':'hi there'})

hi there

In [ ]:
#| export
def transient(data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    display({f'{mimetype}/{subtype}': data}, raw=True, metadata=meta, transient=kw, update=update)

In [ ]:
transient('hi there', foo='bar')

hi there

In [ ]:
transient('*hi* **there**', subtype='markdown')

*hi* **there**

In [ ]:
#| export
def run_cmd(cmd, data='', meta=None, update=False, **kw):
    transient(data, meta=meta, update=update, cmd=cmd, **kw)

## read_url et al

In [ ]:
#| export
def get_md(cts):
    from html2text import HTML2Text
    h2t = HTML2Text(bodywidth=5000)
    h2t.ignore_links = False
    h2t.mark_code = True
    h2t.ignore_images = False
    res = h2t.handle(cts)
    def _f(m): return f'```\n{dedent(m.group(1))}\n```'
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', _f, res or '', flags=re.DOTALL).strip()

In [ ]:
#| export
def scrape_url(url): return create_scraper().get(url)

In [ ]:
#| export
def _get_math_mode():
    v = os.getenv('USE_KATEX', '')
    if v.lower() in {'0', 'false', 'none', ''}: return None
    return 'dollar' if v.lower().startswith('d') else 'safe'

In [ ]:
#| export
def _convert_math(soup, mode):
    for math in soup.find_all('math'):
        annot = math.find('annotation', {'encoding': 'application/x-tex'})
        if not annot: continue
        tex,display = annot.text.strip(), math.get('display') == 'block'
        if mode == 'dollar': wrap = f'$${tex}$$' if display else f'${tex}$'
        else: wrap = f'$${tex}$$' if display else f'\({tex}\)'
        math.replace_with(wrap)

In [ ]:
#| export
def _absolutify_imgs(md, base_url):
    def fix(m):
        alt,img_url = m.group(1),m.group(2)
        if not img_url.startswith('http'): img_url = urljoin(base_url, img_url)
        alt = alt.replace('\\','')
        return f'![{alt}]({img_url})'
    return re.sub(r'!\[(.*?)\]\((.*?)\)', fix, md)

In [ ]:
#| export
def _aify_imgs(md): return re.sub(r'!\[(.*?)\]\((.*?)\)', r'![\1](\2#ai)', md)

In [ ]:
#| export
def read_url(url:str, as_md:bool=True, extract_section:bool=True, selector:str=None, ai_img:bool=False):
    "Read url from web"
    from bs4 import BeautifulSoup
    o = scrape_url(url)
    res,ctype = o.text,o.headers.get('content-type').split(';')[0]
    soup = BeautifulSoup(res, 'lxml')
    
    if selector: res = '\n\n'.join(str(s) for s in soup.select(selector))
    elif extract_section:
        parsed = urlparse(url)
        if parsed.fragment:
            section = soup.find(id=parsed.fragment)
            if section:
                elements = [section]
                current = section.next_sibling
                while current:
                    if hasattr(current, 'name') and current.name == section.name: break
                    elements.append(current)
                    current = current.next_sibling
                res = ''.join(str(el) for el in elements)
            else: res = ''
    else: res = str(soup)
    
    if mmode := _get_math_mode():
        res_soup = BeautifulSoup(res, 'lxml')
        _convert_math(res_soup, mmode)
        res = str(res_soup)
    
    if as_md and ctype == 'text/html':
        h = html2text.HTML2Text()
        h.body_width = 0
        res = _absolutify_imgs(h.handle(res), urljoin(url,s['href'] if (s:=soup.find('base')) else ''))
        if mmode == 'safe': res = res.replace('\\\\(','\\(').replace('\\\\)','\\)')
        
    if ai_img: res = _aify_imgs(res)
    return res

In [ ]:
print(read_url('https://www.example.org'))

# Example Domain

This domain is for use in documentation examples without needing permission. Avoid use in operations.

[Learn more](https://iana.org/domains/example)



In [ ]:
print(read_url('https://www.example.org', as_md=False, selector='body'))

<html><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>


In [ ]:
print(read_url('https://caddyserver.com/docs/running#unit-files'))

### Unit Files

We provide two different systemd unit files that you can choose between, depending on your use case:

  * [**`caddy.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy.service) if you configure Caddy with a [Caddyfile](/docs/caddyfile). If you prefer to use a different config adapter or a JSON config file, you may override the `ExecStart` and `ExecReload` commands.

  * [**`caddy-api.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy-api.service) if you configure Caddy solely through its [API](/docs/api). This service uses the [`--resume`](/docs/command-line#caddy-run) option which will start Caddy using the `autosave.json` which is [persisted](/docs/json/admin/config/) by default.




They are very similar, but differ in the `ExecStart` and `ExecReload` commands to accommodate the workflows.

If you need to switch between the services, you should disable and stop the previous one before enabling and starting the other. For example

`html2text` removes new lines so we only use `get_md` on html content and static text/markdown content is returned as is with new lines preserved:

In [ ]:
print('\n'.join(read_url('https://fastht.ml/docs/llms.txt').splitlines()[:5]))

<html><body># FastHTML

&gt; FastHTML is a python library which brings together Starlette, Uvicorn, HTMX, and fastcore's `FT` "FastTags" into a library for creating server-rendered hypermedia applications. The `FastHTML` class itself inherits from `Starlette`, and adds decorator-based routing with many additions, Beforeware, automatic `FT` to HTML rendering, and much more.

Things to remember when writing FastHTML apps:


read_url now renders latex based on the solveit user's katex setting (`USE_KATEX`)

In [ ]:
os.environ['USE_KATEX']='dollar'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1'))

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure [2](https://arxiv.org/html/1706.03762v7#S3.F2 "Figure 2 ‣ 3.2.2 Multi-Head Attention ‣ 3.2 Attention ‣ 3 Model Architecture ‣ Attention Is All You Need")). The input consists of queries and keys of dimension $d_{k}$, and values of dimension $d_{v}$. We compute the dot products of the query with all keys, divide each by $\sqrt{d_{k}}$, and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix $Q$. The keys and values are also packed together into matrices $K$ and $V$. We compute the matrix of outputs as:

| $$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(\frac{QK^{T}}{\sqrt{d_{k}}})V$$ |  | (1)  
---|---|---|---  
  
The two most commonly used attention functions are additive attention [[2](https://arxiv.org/html/1706.03762v7#bib.bib2)], and dot-product (mul

In [ ]:
os.environ['USE_KATEX']='1'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1'))

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure [2](https://arxiv.org/html/1706.03762v7#S3.F2 "Figure 2 ‣ 3.2.2 Multi-Head Attention ‣ 3.2 Attention ‣ 3 Model Architecture ‣ Attention Is All You Need")). The input consists of queries and keys of dimension \(d_{k}\), and values of dimension \(d_{v}\). We compute the dot products of the query with all keys, divide each by \(\sqrt{d_{k}}\), and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix \(Q\). The keys and values are also packed together into matrices \(K\) and \(V\). We compute the matrix of outputs as:

| $$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(\frac{QK^{T}}{\sqrt{d_{k}}})V$$ |  | (1)  
---|---|---|---  
  
The two most commonly used attention functions are additive attention [[2](https://arxiv.org/html/1706.03762v7#bib.bib2)], and dot-

Relative image paths are automatically corrected as well:

In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1'))

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/1706.03762v7/x1.png) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x2.png)

![Refer to caption](https://arxiv.org/html/1706.03762v7/x3.png)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and 6. Note that the attentions are very sharp for this word.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x4.png)

![Refer to caption](https://arxiv.org/html/1706.03762v7/x5.png)

Fi

In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1',ai_img=True))

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/1706.03762v7/x1.png#ai) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x2.png#ai)

![Refer to caption](https://arxiv.org/html/1706.03762v7/x3.png#ai)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and 6. Note that the attentions are very sharp for this word.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x4.png#ai)

![Refer to caption](https://arxiv.org/html/1706.03762v7

## Extension

In [ ]:
#| export
@patch
def _get_info(self:Inspector, obj, oname='', formatter=None, info=None, detail_level=0, omit_sections=()):
    "Custom formatter for ?? output"
    orig = self._orig__get_info(obj, oname=oname, formatter=formatter, info=info,
                               detail_level=detail_level, omit_sections=omit_sections)
    if detail_level==0: return orig
    info_dict = self.info(obj, oname=oname, info=info, detail_level=detail_level)
    out = []
    if c:=info_dict.get('source'): out.append(f"\n```python\n{dedent(c)}\n```")
    if c:=info_dict.get('file'): out.append(f"**File:** `{c}`")
    return {'text/markdown': '\n\n'.join(out), 'text/html': '', 'text/plain': orig['text/plain']}

In [ ]:
patch?

Signature: patch(f=None, *, as_prop=False, cls_method=False, set_prop=False)
Docstring: Decorator: add `f` to the first parameter's class (based on f's type annotations)
File:      ~/git/repos/fastcore/fastcore/basics.py
Type:      function

In [ ]:
patch??


```python
def patch(f=None, *, as_prop=False, cls_method=False,  set_prop=False):
    "Decorator: add `f` to the first parameter's class (based on f's type annotations)"
    if f is None: return partial(patch, as_prop=as_prop, cls_method=cls_method, set_prop=set_prop)
    ann,glb,loc = get_annotations_ex(f)
    cls = union2tuple(eval_type(ann.pop('cls') if cls_method else next(iter(ann.values())), glb, loc))
    return patch_to(cls, as_prop=as_prop, cls_method=cls_method, set_prop=set_prop)(f)
```

**File:** `~/git/repos/fastcore/fastcore/basics.py`

In [ ]:
#| export
def load_ipython_extension(ip):
    from ipykernel_helper import transient,run_cmd
    ns = ip.user_ns
    ns['read_url'],ns['transient'],ns['run_cmd'] = read_url,transient,run_cmd

## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()